# 1D J1-J2=0.0-J3=0.5: LorentzGRU (with manifold update)

This notebook is part of the work https://arxiv.org/abs/2604.24337. 

In [1]:
import os
import sys
sys.path.append('../utility_lorentz')
from j1j2j3_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
E_exact =-53.99140745384424
units = 80
syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.0
J3 = 0.5
lr1=5e-3
lr2=5e-3

nsteps = 1201
var_tol = 20.0

In [3]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

# LorentzGRU

Note that for LorentzGRU (as well as LorentzRNN), the last parameter k is not a learnable parameter. 

## Single clamp in forward pass

In [4]:
spatial_clamp=4.0
units = 70
fname=f'1d_J1J2J3_results_LorentzGRU_N=100_manifold_update_single_clamp/J2={J2}_J3={J3}'
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units, 
                       spatial_clamp=spatial_clamp,seed=111)
for name, param in wf.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params ={sum(p.numel() for p in wf.model.parameters())}')

t0=time.time()
mE, vE = run_J1J2J3(wf, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=8e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.Wz | Size: torch.Size([70, 70])
Layer: rnn.Uz | Size: torch.Size([70, 2])
Layer: rnn.bz | Size: torch.Size([70])
Layer: rnn.Wr | Size: torch.Size([70, 70])
Layer: rnn.Ur | Size: torch.Size([70, 2])
Layer: rnn.br | Size: torch.Size([70])
Layer: rnn.Wh | Size: torch.Size([70, 70])
Layer: rnn.Uh | Size: torch.Size([70, 2])
Layer: rnn.bh | Size: torch.Size([70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])
Total params =15615


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.99316, mean energy: 36.45669+0.06355j|varE: 1.39076| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.05
max_h0 = 3.6054399013519287 | max_h_spatial_norm = 3.4639856815338135| max_h_violation = 5.7220458984375e-06
Best model saved at epoch 1 with best E=35.11857-0.11830j, varE=5.72612
step: 10, loss: 16.03059, mean energy: 0.87947-0.27126j|varE: 25.20575| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.0490000000000002
max_h0 = 3.3590383529663086 | max_h_spatial_norm = 3.206733465194702| max_h_violation = 3.814697265625e-06
step: 20, loss: -11.72473, mean energy: -2.86851+0.48902j|varE: 26.98637| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.048
max_h0 = 3.2680954933166504 | max_h_spatial_norm = 3.111341953277588| max_h_violation = 2.86102294921875e-06
step: 30, loss: 2.27161, mean energy: -8.33474+0.27100j|varE: 27.29282| Hyp LR: 8.00e-03| LR: 5.00e-03| tau=1.0470000000000002
max_h0 = 3.3345510959625244 | max_h_spatial_norm = 3.1810736656188965| max_h_violation = 2.86102294921875e-06
step: 40, l